In [ ]:
import numpy as np 
#want to generate U, V randomly. Initialize each entry to be drawn from a normal independently. 
#U has dimensions n x r
n = 100
r = 10
p = 500
U_true = np.random.normal(0,1,size=(n,r))
V_true = np.random.normal(0,1,size=(p,r))
Theta_true = np.matmul(U_true,V_true.T) #shape: (n,p)



In [20]:
P_true = 1/(1+np.exp(-Theta_true)) #shape: (n,p)
Y_obs = np.random.binomial(1,P_true)
#let g1,g2 have n/2 samples each. Let g1 unique features be the 
# first floor(p/3) cols, g2 unique features be the second floor(p/3) cols, shared features are the last p-2*floor(p/3) cols
g1_unique = Y_obs[:n // 2, :p // 3]
g1_shared = Y_obs[:n // 2, 2*(p//3):]
g2_unique = Y_obs[n // 2:, p // 3:2*(p//3)]
g2_shared = Y_obs[n // 2:,2*(p//3):]
Y_g1 = np.concatenate([g1_unique,g1_shared],axis= 1)
Y_g2 = np.concatenate([g2_unique,g2_shared],axis=1)
Y_shared = np.concatenate([g1_shared,g2_shared],axis=0)

In [24]:
U_g1=np.random.normal(0,0.01,size=(n // 2,r))
V_g1 = np.random.normal(0,0.01,size=(p - p // 3,r))
U_g2 = np.random.normal(0,0.01,size=(n - n//2,r))
V_g2 = np.random.normal(0,0.01,size=(p - p // 3,r))


In [25]:
def nll(U,V,Y):
    Theta = np.matmul(U,V.T)
    inside = np.logaddexp(np.zeros(Theta.shape),Theta)
    summand = inside - Y*Theta
    return np.sum(summand)
def sigmoid(x):
    return 1/(1+np.exp(-x))
def s1_fit(Y_g,lr,tol,max_iters,U_g,V_g):
    iters = 0
    prev_nll = float('inf')
    curr_nll = nll(U_g,V_g,Y_g)
    while abs(curr_nll-prev_nll) > tol and iters <= max_iters:
        R_g = sigmoid(np.matmul(U_g,V_g.T))-Y_g
        u_update = lr*np.matmul(R_g,V_g)
        v_update = lr*np.matmul(R_g.T,U_g)
        U_g,V_g = U_g-u_update,V_g-v_update
        prev_nll = curr_nll
        curr_nll = nll(U_g,V_g,Y_g)
        iters += 1
    return U_g,V_g


    



In [26]:
U_g1s1,V_g1s1 = s1_fit(Y_g1,0.001,10**-6,10000,U_g1,V_g1)
U_g2s1,V_g2s1=s1_fit(Y_g2,0.001,10**-6,10000,U_g2,V_g2)

In [ ]:
Q_g1,_ = np.linalg.qr(U_g1s1,mode="reduced")
Q_g2,_ = np.linalg.qr(U_g2s1,mode="reduced")
print(Q_g1.shape,Q_g2.shape)
Q_hat = np.concatenate([np.concatenate([Q_g1,np.zeros((Q_g1.shape[0],Q_g2.shape[1]))],axis=1),np.concatenate([np.zeros((Q_g2.shape[0],Q_g1.shape[1])),Q_g2],axis=1)],axis=0)


(50, 10) (50, 10)
[[-0.01080262  0.0813089   0.09340017 ...  0.          0.
   0.        ]
 [-0.24634085 -0.13833106  0.19059307 ...  0.          0.
   0.        ]
 [ 0.00838549 -0.00155826 -0.00589658 ...  0.          0.
   0.        ]
 ...
 [ 0.          0.          0.         ...  0.04917418  0.13409498
  -0.16227958]
 [ 0.          0.          0.         ... -0.00813226 -0.00501406
  -0.00480524]
 [ 0.          0.          0.         ...  0.00855906  0.00252751
  -0.00433834]]


In [34]:
proj = np.matmul(Q_hat,Q_hat.T)
U_hat = np.random.normal(0,0.01,size=(n,r))
V_hat = np.random.normal(0,0.01,size=(p-2*(p // 3),r))
def s2_fit(U,V,Y,lr,tol,max_iters,proj):
    U = np.matmul(proj,U)
    iters = 0
    prev_nll = float('inf')
    curr_nll = nll(U,V,Y)
    while abs(curr_nll-prev_nll) >= tol and iters < max_iters:
        R_hat = sigmoid(np.matmul(U,V.T))-Y
        u_update = lr*np.matmul(R_hat,V)
        v_update = lr*np.matmul(R_hat.T,U)
        U,V = U - u_update,V - v_update
        U = np.matmul(proj,U)
        iters += 1
        prev_nll = curr_nll
        curr_nll = nll(U,V,Y)
    return U,V
U_final,V_final = s2_fit(U_hat,V_hat,Y_shared,0.01,10**-6,10000,proj)
    
    